# Thursday add-on: why does least-squares break, and what do you do about it?

ASTR 457, Fall 2026 — pulled forward from the original Sep 1/3 slot.

Ordinary least-squares fitting assumes:
- your `y` uncertainties are Gaussian
- your `x` values are essentially exact
- every point's error is independent of every other point's

Astronomical data routinely violates all three. Today: what breaks, and two
standard fixes:
- **k-sigma clipping** — what Lab 01 Part 2 has you implement
- **a mixture model** — what Lab 01 Part 3 grades you on understanding, and
  what we're going to actually build together, not just describe

*Adapted from LSSTC-DSFP Session 7 Day0 "The Assumptions of Least Squares"
(public, CC-licensed teaching material) — reworked here as a live-taught demo
rather than a fill-in-the-blank problem set.*

## A small dataset with a big problem

- 20 brightness measurements of a source, each with a stated uncertainty
- some of them are contaminated — bad pixels, cosmic-ray hits, a satellite
  trail, take your pick
- you don't know which ones

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260827)

n = 20
x_true = np.linspace(0, 10, n)
y_true = 2.0 * x_true + 5.0
sigma = np.full(n, 1.5)
y = y_true + rng.normal(0, sigma)

# contaminate 4 points with a much larger, non-Gaussian excursion
bad = rng.choice(n, size=4, replace=False)
y[bad] += rng.choice([-1, 1], size=4) * rng.uniform(8, 15, size=4)

plt.errorbar(x_true, y, yerr=sigma, fmt='o', capsize=3)
plt.xlabel('x'); plt.ylabel('y')
plt.title('Raw data — which points do you trust?')
plt.show()

**Ask the room:** just by eye, which points look wrong? Write down which
indices you'd throw out before running any code.

## Fit 1: ordinary least squares, everything included

This is what `np.polyfit` (or an AI assistant's first pass) hands you if you
don't say anything about outliers.

In [ ]:
p_naive = np.polyfit(x_true, y, 1, w=1/sigma)
resid = y - np.polyval(p_naive, x_true)
chi2 = np.sum((resid / sigma) ** 2)
chi2_red = chi2 / (n - 2)
print(f"naive fit: slope={p_naive[0]:.2f}, intercept={p_naive[1]:.2f}, "
      f"chi2_red={chi2_red:.2f}  (true slope=2.00, intercept=5.00)")

xx = np.linspace(0, 10, 200)
plt.errorbar(x_true, y, yerr=sigma, fmt='o', capsize=3, label='data')
plt.plot(xx, np.polyval(p_naive, xx), 'r-', label='naive fit')
plt.plot(xx, 2.0 * xx + 5.0, 'k--', alpha=0.5, label='truth')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.title(f'Naive fit — chi2_red = {chi2_red:.1f}')
plt.show()

- chi2_red is way above 1 — same warning sign as Tuesday's demo
- the contaminated points are dragging both the slope and the intercept
  away from truth
- and they're dragging your uncertainty estimate too: a least-squares fit
  reports smaller error bars than it should, because it doesn't know four
  of its inputs are lying to it

## Fit 2: k-sigma clipping

- fit once, throw out anything more than $k\sigma$ from the fit, refit
- iterate until nothing new gets clipped
- this is what Lab 01 Part 2 asks you to implement

In [ ]:
def sigma_clip_fit(x, y, sigma, k=3.0, max_iter=10):
    mask = np.ones_like(x, dtype=bool)
    for _ in range(max_iter):
        p = np.polyfit(x[mask], y[mask], 1, w=1/sigma[mask])
        resid = (y - np.polyval(p, x)) / sigma
        new_mask = np.abs(resid) < k
        if np.array_equal(new_mask, mask):
            break
        mask = new_mask
    return p, mask

p_clip, mask = sigma_clip_fit(x_true, y, sigma, k=3.0)
resid = y[mask] - np.polyval(p_clip, x_true[mask])
chi2_red_clip = np.sum((resid / sigma[mask]) ** 2) / (mask.sum() - 2)
print(f"3-sigma clip: slope={p_clip[0]:.2f}, intercept={p_clip[1]:.2f}, "
      f"chi2_red={chi2_red_clip:.2f}, clipped {(~mask).sum()} of {n} points")
print(f"actually-contaminated indices: {sorted(bad)}; clipped indices: {sorted(np.where(~mask)[0])}")

plt.errorbar(x_true[mask], y[mask], yerr=sigma[mask], fmt='o', capsize=3, color='C0', label='kept')
plt.errorbar(x_true[~mask], y[~mask], yerr=sigma[~mask], fmt='x', capsize=3, color='r', label='clipped')
plt.plot(xx, np.polyval(p_naive, xx), 'r-', alpha=0.4, label='naive fit')
plt.plot(xx, np.polyval(p_clip, xx), 'C0-', label='clipped fit')
plt.plot(xx, 2.0 * xx + 5.0, 'k--', alpha=0.5, label='truth')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.title(f'3-sigma clip — chi2_red = {chi2_red_clip:.2f}')
plt.show()

**The problem with k-sigma clipping:** the threshold $k$ is a knob you
picked, not something the data derived for you. Two failure modes, both real
— let's actually see them, not just take my word for it.

## When your stated uncertainties are themselves wrong

- if $\sigma$ is **underestimated**, clipping throws away good data — points
  that are honestly a bit noisy get flagged as outliers
- if $\sigma$ is **overestimated**, clipping keeps bad data — genuine
  contaminants no longer look like enough of a statistical outlier to clip
- **the trap:** from the fit alone, both regimes can look "clean" (low
  chi2_red) — you cannot tell which one you're in without independent
  knowledge of your uncertainties

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, factor, label in zip(axes, [0.3, 3.0], ['sigma underestimated (x0.3)', 'sigma overestimated (x3.0)']):
    sigma_wrong = sigma * factor
    p_w, mask_w = sigma_clip_fit(x_true, y, sigma_wrong, k=3.0)
    r_w = y[mask_w] - np.polyval(p_w, x_true[mask_w])
    cr_w = np.sum((r_w / sigma_wrong[mask_w]) ** 2) / (mask_w.sum() - 2)
    n_true_bad_kept = len(set(np.where(~mask_w)[0]) & set(bad))

    ax.errorbar(x_true[mask_w], y[mask_w], yerr=sigma_wrong[mask_w], fmt='o', capsize=3, color='C0', label='kept')
    ax.errorbar(x_true[~mask_w], y[~mask_w], yerr=sigma_wrong[~mask_w], fmt='x', capsize=3, color='r', label='clipped')
    ax.plot(xx, np.polyval(p_w, xx), 'C0-', label='fit')
    ax.plot(xx, 2.0 * xx + 5.0, 'k--', alpha=0.5, label='truth')
    ax.set_title(f'{label}\nclipped {(~mask_w).sum()}/{n}, chi2_red={cr_w:.2f}\n{n_true_bad_kept}/4 real contaminants still kept in')
    ax.set_xlabel('x')
axes[0].set_ylabel('y'); axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Try it live** — instead of retyping `k=` and rerunning, drag the slider.
Watch which points get clipped (and the fit quality) change as you sweep $k$
from aggressive to permissive.

In [ ]:
from ipywidgets import interact, FloatSlider

@interact(k=FloatSlider(value=3.0, min=1.0, max=6.0, step=0.25, description='k (sigma)'))
def explore_clipping(k):
    p, m = sigma_clip_fit(x_true, y, sigma, k=k)
    r = y[m] - np.polyval(p, x_true[m])
    cr = np.sum((r / sigma[m]) ** 2) / (m.sum() - 2) if m.sum() > 2 else np.nan

    plt.figure(figsize=(6, 4))
    plt.errorbar(x_true[m], y[m], yerr=sigma[m], fmt='o', capsize=3, color='C0', label='kept')
    plt.errorbar(x_true[~m], y[~m], yerr=sigma[~m], fmt='x', capsize=3, color='r', label='clipped')
    plt.plot(xx, np.polyval(p, xx), 'C0-', label='fit')
    plt.plot(xx, 2.0 * xx + 5.0, 'k--', alpha=0.5, label='truth')
    plt.xlabel('x'); plt.ylabel('y'); plt.legend(loc='upper left')
    plt.title(f'k={k:.2f}: clipped {(~m).sum()} of {n} points, chi2_red={cr:.2f}')
    plt.show()

## The better way: a mixture model (Hogg, Bovy & Lang 2010)

Same paper Lab 01 already points you to. Instead of a hard in/out decision,
model the data as coming from **two** populations at once:

- a "good" population: Gaussian around the line, with the stated $\sigma_i$
- a "bad" population: its own Gaussian, mean $Y_b$ and variance $V_b$ —
  **not tied to the line at all**
- every point gets a *probability* $P_i$ of being bad, fit from the data —
  nothing is manually thrown away, and the outlier fraction $P_b$ becomes a
  parameter you estimate instead of a knob you turn

This is exactly what Lab 01 Part 3 asks you to build, and it's why your
calibration grade there depends on getting each point's outlier probability
right, not just the slope. Let's fit one live.

In [ ]:
from scipy.optimize import minimize
from scipy.stats import norm

def neg_log_likelihood(theta, x, y, sigma):
    m, b, Pb, Yb, lnVb = theta
    Pb = np.clip(Pb, 1e-6, 1 - 1e-6)
    Vb = np.exp(lnVb)
    model = m * x + b
    good = (1 - Pb) * norm.pdf(y, loc=model, scale=sigma)
    bad_pop = Pb * norm.pdf(y, loc=Yb, scale=np.sqrt(Vb + sigma**2))
    return -np.sum(np.log(good + bad_pop + 1e-300))

# start from the sigma-clip fit — a sensible, defensible initial guess
theta0 = [p_clip[0], p_clip[1], 0.2, np.mean(y), np.log(50.0)]
result = minimize(neg_log_likelihood, theta0, args=(x_true, y, sigma), method='Nelder-Mead')
m_mix, b_mix, Pb_mix, Yb_mix, lnVb_mix = result.x
print(f"mixture fit: slope={m_mix:.2f}, intercept={b_mix:.2f}, "
      f"outlier fraction Pb={Pb_mix:.2f} (true fraction = {len(bad)/n:.2f})")

In [ ]:
# per-point posterior probability of belonging to the "bad" population
Vb_mix = np.exp(lnVb_mix)
model_mix = m_mix * x_true + b_mix
good_l = (1 - Pb_mix) * norm.pdf(y, loc=model_mix, scale=sigma)
bad_l = Pb_mix * norm.pdf(y, loc=Yb_mix, scale=np.sqrt(Vb_mix + sigma**2))
p_bad = bad_l / (good_l + bad_l)

plt.scatter(x_true, y, c=p_bad, cmap='coolwarm', s=80, edgecolor='k', vmin=0, vmax=1)
plt.colorbar(label='posterior P(outlier)')
plt.plot(xx, m_mix * xx + b_mix, 'g-', label='mixture fit')
plt.plot(xx, np.polyval(p_clip, xx), 'C0--', alpha=0.6, label='sigma-clip fit')
plt.plot(xx, 2.0 * xx + 5.0, 'k:', alpha=0.5, label='truth')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.title('Mixture model: color = probability this point is an outlier')
plt.show()

print("point-by-point P(outlier):")
for i in np.argsort(-p_bad)[:6]:
    flag = 'REAL contaminant' if i in bad else 'clean'
    print(f"  point {i}: P(outlier)={p_bad[i]:.2f}  [{flag}]")

**What just happened, and why it's better:**

- no point got permanently thrown away — every point still has *some* weight
  in the fit, proportional to how likely it is to be good
- the outlier fraction $P_b$ came out of the fit, not from a threshold you
  picked
- compare the printed probabilities to the actually-contaminated indices
  above — high-probability points should mostly be real contaminants, and
  the fit degrades gracefully even where the classification is uncertain,
  instead of making a hard, unexplainable cut

## Bridge to Lab 01

- **sigma-clipping** is the version you can write in five minutes and defend
  in one sentence — good enough when the answer isn't close to the
  threshold
- **the mixture model** is the version that doesn't require you to
  pre-commit to a threshold, reports an outlier fraction as a real
  parameter, and degrades gracefully instead of making hard cuts
- you'll implement **both** in Lab 01 — now you've seen why the second one
  earns the extra 20 minutes it costs you